# 06 — Benchmarking the CAA implementation against Rimsky et al. (2024)

Runs Rimsky et al.'s released Llama-2-7B-Chat protocol through this repo's code: their vector through our hook, and our vector extraction on their data.

In [1]:
# Environment: on Colab clone the repo, install requirements and mount Drive; then build the pairs
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/chase-mayer/syco-erase.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/syco-erase").exists():
        subprocess.run(["git", "clone", "-q", REPO, "/content/syco-erase"], check=True)
    os.chdir("/content/syco-erase")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
assert Path("src/experiment.py").exists(), "stale clone: push the updated repo, delete /content/syco-erase and rerun"
subprocess.run(["bash", "scripts/download_data.sh"], check=True, capture_output=True)
print(subprocess.run([sys.executable, "-m", "src.data", "--dataset", "all"],
                     check=True, capture_output=True, text=True).stdout)

Mounted at /content/drive
nlp: 9984 pairs, 32 unique questions -> pairs_nlp.jsonl  (stem match 100.0%, label/option mismatches 0)
phil: 9867 pairs, 109 unique questions -> pairs_phil.jsonl  (stem match 100.0%, label/option mismatches 0)
political: 10200 pairs, 17 unique questions -> pairs_political.jsonl  (stem match 100.0%, label/option mismatches 0)



In [2]:
# Hugging Face login for the gated Llama-2 weights (skipped in offline smoke tests)
if os.environ.get("BES_SMOKE") != "1":
    from huggingface_hub import login
    login()

In [3]:
# Load Llama-2-7B-Chat through the same Runner used for Qwen
import glob, json, re
import numpy as np
import pandas as pd
import torch
from src import bes, experiment as E, plots

cfg = E.Config(model=os.environ.get("BES_LLAMA_MODEL", "meta-llama/Llama-2-7b-chat-hf"), batch_size=8, dtype="fp16")
run = E.Runner(cfg)
LAYER = 13 if not cfg.smoke else 1
print(cfg.model, "| layers", run.n_layers, "| steering layer", LAYER)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

meta-llama/Llama-2-7b-chat-hf | layers 32 | steering layer 13


In [4]:
# Rimsky et al.'s released data, vectors and per-item results (MIT licence)
ref = Path("data/raw/caa_ref")
if not ref.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nrimsky/CAA.git", str(ref)], check=True)
gen = json.load(open(ref / "datasets/generate/sycophancy/generate_dataset.json"))
test = json.load(open(ref / "datasets/test/sycophancy/test_dataset_ab.json"))
v_pub = torch.load(ref / "normalized_vectors/sycophancy/vec_layer_13_Llama-2-7b-chat-hf.pt").float()
v_pub_raw = torch.load(ref / "vectors/sycophancy/vec_layer_13_Llama-2-7b-chat-hf.pt").float()
if cfg.smoke:
    v_pub_raw = torch.randn(run.d_model)
    v_pub = 4 * v_pub_raw
published = {}
for f in glob.glob(str(ref / "results/sycophancy/results_layer=13_multiplier=*_behavior=sycophancy_type=ab_use_base_model=False_model_size=7b.json")):
    mult = float(re.search(r"multiplier=([-\d.]+)_", f).group(1))
    rs = json.load(open(f))
    published[mult] = np.mean([r["a_prob"] if r["answer_matching_behavior"].strip() == "(A)" else r["b_prob"] for r in rs])
print(len(gen), "generation pairs |", len(test), "test items | published norm", round(float(v_pub.norm()), 3),
      "| raw norm", round(float(v_pub_raw.norm()), 3))
print("published P(sycophantic letter):", {k: round(float(v), 3) for k, v in sorted(published.items())})

1000 generation pairs | 50 test items | published norm 7.235 | raw norm 1.755
published P(sycophantic letter): {-1.0: 0.355, -0.5: 0.439, 0.0: 0.562, 0.5: 0.618, 1.0: 0.601}


In [5]:
# Llama-2 chat formatting and the steering span (from the last token of [/INST] to the end)
def chat(q, out):
    return f"[INST] {q.strip()} [/INST] {out}"


def from_inst_end(prompts):
    spans = []
    for p in prompts:
        head = p[: p.rindex("[/INST]") + len("[/INST]")]
        spans.append((len(run.tok.encode(p)) - len(run.tok.encode(head)) + 1, 0))
    return spans


prompts = [chat(t["question"], "(") for t in test]
match = np.array([bes.LETTERS.index(t["answer_matching_behavior"].strip()[1]) for t in test])
spans = from_inst_end(prompts)


def p_syco(vector, mult):
    iv = None if mult == 0 else bes.AddVector(layers=[LAYER], vector=vector, alpha=mult)
    p = bes.option_probs(run.model, run.tok, prompts, [["A", "B"]] * len(prompts), run.device,
                         iv=iv, spans=spans, batch_size=cfg.batch_size, normalize=False)
    return float(p[np.arange(len(p)), match].mean())

In [6]:
# Check 1: their vector through our hook and scoring
MULTS = sorted(published) or [-1.0, 0.0, 1.0]
check1 = pd.DataFrame([{"multiplier": m, "published": published.get(m, np.nan), "ours_their_vector": p_syco(v_pub, m)} for m in MULTS])
check1["abs_diff"] = (check1.ours_their_vector - check1.published).abs()
check1.round(4)

,multiplier,published,ours_their_vector,abs_diff
0,-1.0,0.3549,0.3560,0.0012
1,-0.5,0.4393,0.4407,0.0013
2,0.0,0.5623,0.5626,0.0003
3,0.5,0.6182,0.6182,0.0000
4,1.0,0.6008,0.6018,0.0010


In [7]:
# Check 2: our CAA extraction on their 1,000 generation pairs (activation at the answer letter, position -2)
def letter_acts(answers):
    texts = [chat(g["question"], a) for g, a in zip(gen, answers)]
    h = bes.suffix_activations(run.model, run.tok, texts, [LAYER], 2, run.device, cfg.batch_size)[LAYER]
    return h.reshape(len(texts), 2, -1)[:, 0]


v_ours_raw = bes.caa_vector(letter_acts([g["answer_matching_behavior"] for g in gen]),
                            letter_acts([g["answer_not_matching_behavior"] for g in gen]))
v_ours = v_ours_raw * (v_pub.norm() / v_ours_raw.norm())
cos = float(torch.nn.functional.cosine_similarity(v_ours_raw, v_pub_raw, dim=0))
check2 = pd.DataFrame([{"multiplier": m, "published": published.get(m, np.nan), "ours_our_vector": p_syco(v_ours, m)} for m in MULTS])
check2["abs_diff"] = (check2.ours_our_vector - check2.published).abs()
print(f"cos(our raw vector, their raw vector) = {cos:.4f} | norms: ours {float(v_ours_raw.norm()):.3f}, theirs {float(v_pub_raw.norm()):.3f}")
check2.round(4)

cos(our raw vector, their raw vector) = 1.0000 | norms: ours 1.755, theirs 1.755


,multiplier,published,ours_our_vector,abs_diff
0,-1.0,0.3549,0.3559,0.0010
1,-0.5,0.4393,0.4406,0.0012
2,0.0,0.5623,0.5626,0.0003
3,0.5,0.6182,0.6183,0.0001
4,1.0,0.6008,0.6020,0.0012


In [8]:
# Save the reproduction table
repro = check1.merge(check2[["multiplier", "ours_our_vector"]], on="multiplier")
repro["cos_vectors"] = cos
E.save(repro, "caa_reproduction_llama2")
repro.round(4)

,multiplier,published,ours_their_vector,abs_diff,ours_our_vector,cos_vectors
0,-1.0,0.3549,0.3560,0.0012,0.3559,1.0
1,-0.5,0.4393,0.4407,0.0013,0.4406,1.0
2,0.0,0.5623,0.5626,0.0003,0.5626,1.0
3,0.5,0.6182,0.6182,0.0000,0.6183,1.0
4,1.0,0.6008,0.6018,0.0010,0.6020,1.0
